In [1]:
# imports 
import ast
import difflib
import hashlib
import json
import math
import os
import re
import unicodedata
from pathlib import Path
from typing import Annotated
from urllib.parse import quote_plus
from difflib import SequenceMatcher
from functools import lru_cache

import numexpr
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

import ifcopenshell

In [2]:
def load_llm(id_model, temperature):
    llm = ChatOpenAI(
        model=id_model,
        temperature=temperature,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    return llm

In [ ]:
openai_api_key = os.environ.get("OPENAI_API_KEY") or os.environ.get("openai_api_key")
if not openai_api_key:
    raise EnvironmentError("Defina OPENAI_API_KEY ou openai_api_key antes de criar o LLM.")
os.environ["OPENAI_API_KEY"] = openai_api_key
id_model = "gpt-4.1"
temperature = 0.2

llm = load_llm(id_model, temperature)

In [ ]:
# Math tools
@tool
def calculator_tool(expression: str) -> str:
    """Use para qualquer calculo aritmetico direto; não faça cálculo numérico de cabeça."""
    local_dict = {"pi": math.pi, "e": math.e}
    return str(
        numexpr.evaluate(
            expression.strip(),
            global_dict={},
            local_dict=local_dict,
        )
    )


@tool
def percentage_tool(base_value: float, percent: float) -> dict:
    """Use para calcular porcentagem aplicada sobre um valor base; não faça cálculo de cabeça."""
    percentage_value = base_value * (percent / 100)
    total_with_percentage = base_value + percentage_value
    return {
        "base_value": base_value,
        "percent": percent,
        "percentage_value": percentage_value,
        "total_with_percentage": total_with_percentage,
    }


@tool
def percent_change_tool(initial_value: float, final_value: float) -> dict:
    """Use para calcular variação percentual entre dois valores; não faça cálculo de cabeça."""
    if initial_value == 0:
        return {
            "error": "initial_value não pode ser zero para calcular variação percentual."
        }

    absolute_change = final_value - initial_value
    percent_change = (absolute_change / initial_value) * 100
    return {
        "initial_value": initial_value,
        "final_value": final_value,
        "absolute_change": absolute_change,
        "percent_change": percent_change,
    }

In [ ]:
# Supplier models and placeholder search tool
import sys
from datetime import date
from typing import Any

from langchain_core.messages import HumanMessage, SystemMessage

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "backend/api/src/app/models/materials.py").exists():
    project_root = project_root.parent

api_src_path = project_root / "backend/api/src"
if str(api_src_path) not in sys.path:
    sys.path.insert(0, str(api_src_path))

from app.models.materials import ListaMateriaisObra, MaterialObra, OfertaFornecedor


@tool
def search_supplier_serp_tool(
    product_name: str,
    unit: str = "",
    quantity: float | None = None,
    profile: str = "Medio custo",
    city: str = "Aracaju - SE"
) -> list[dict]:
    """Search for suppliers of a product using SERP (Search Engine Results Page) API based on the product name, unit, quantity, profile and city"""
    return [
        {
            "status": "tool_not_implemented",
            "product_name": product_name,
            "unit": unit,
            "quantity": quantity,
            "profile": profile,
            "city": city,
            "message": "Define supplier search tools and pass them to build_supplier_reasoning_agent().",
        }
    ]


In [ ]:
# ReAct reasoning agent: ListaMateriaisObra -> ListaMateriaisObra with supplier options
class SupplierReasoningState(TypedDict):
    messages: Annotated[list, add_messages]


SUPPLIER_REASONING_SYSTEM_PROMPT = """
You are the base Reasoning agent for supplier discovery in Obra Barata.
Your job is to control the available tools, search supplier offers for one material at a time,
compare the evidence, and return only a JSON object that can update MaterialObra.

Rules:
- Use tools before choosing suppliers whenever a search tool is available.
- Use tools to make calculations, conversions, and percentage computations; do not calculate in your head, when possible.
- Do not invent suppliers, prices, links, freight, installments, or availability.
- If the available tool is only a placeholder or returns no offers, return an empty lista_fornecedores.
- Prefer offers that match product name, unit, quantity, and product profile.
- Return JSON only, with this shape:
{
  "fornecedor": "best supplier name or empty string",
  "lista_fornecedores": [
    {
      "fornecedor": "supplier name",
      "descricao": "product description",
      "marca": "brand or null",
      "unidade": "commercial unit",
      "quantidade": 1,
      "valor_unitario": 0,
      "valor_total": 0,
      "preco_a_vista": 0,
      "preco_a_prazo": 0,
      "num_parcelas": 1,
      "frete": 0,
      "disponibilidade": "availability text",
      "data_consulta": "YYYY-MM-DD",
      "link_produto": "source URL"
    }
  ],
  "valor_unitario": 0,
  "valor_total": 0,
  "preco_a_vista": 0,
  "preco_a_prazo": 0,
  "num_parcelas": 1,
  "frete": 0,
  "justificativa": "short evidence-based explanation"
}
""".strip()


def build_supplier_reasoning_agent(
    reasoning_llm,
    supplier_search_tools: list | None = None,
    checkpointer: MemorySaver | None = None,
):
    """Build the base ReAct graph that controls supplier search tools."""
    tools = list(supplier_search_tools)
    llm_with_tools = reasoning_llm.bind_tools(tools)

    def reasoning_agent(state: SupplierReasoningState) -> dict:
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    graph = StateGraph(SupplierReasoningState)
    graph.add_node("reasoning_agent", reasoning_agent)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "reasoning_agent")
    graph.add_conditional_edges("reasoning_agent", tools_condition)
    graph.add_edge("tools", "reasoning_agent")
    return graph.compile(checkpointer=checkpointer)


def _message_text(message) -> str:
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


def _extract_json_object(text: str) -> dict:
    fenced_match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fenced_match:
        return json.loads(fenced_match.group(1))

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return {"lista_fornecedores": [], "justificativa": text.strip()}
    return json.loads(text[start : end + 1])


def _material_prompt(area_name: str, material: MaterialObra) -> str:
    payload = {
        "area": area_name,
        "material": material.model_dump(mode="json"),
        "data_consulta": date.today().isoformat(),
    }
    return (
        "Find supplier offers for this MaterialObra and return the JSON update only.\n"
        + json.dumps(payload, ensure_ascii=False, separators=(",", ":"))
    )


def _valid_offer_payloads(update_payload: dict) -> list[OfertaFornecedor]:
    raw_offers = update_payload.get("lista_fornecedores") or update_payload.get("ofertas") or []
    offers = []
    for raw_offer in raw_offers:
        if not isinstance(raw_offer, dict) or raw_offer.get("status") == "tool_not_implemented":
            continue
        try:
            offers.append(OfertaFornecedor.model_validate(raw_offer))
        except Exception as exc:
            print(f"Skipping invalid supplier offer: {exc}")
    return offers


def _best_offer(offers: list[OfertaFornecedor]) -> OfertaFornecedor | None:
    priced_offers = [offer for offer in offers if offer.valor_unitario is not None]
    if priced_offers:
        return min(priced_offers, key=lambda offer: offer.valor_unitario or float("inf"))
    return offers[0] if offers else None


def _apply_supplier_update(material: MaterialObra, update_payload: dict) -> MaterialObra:
    offers = _valid_offer_payloads(update_payload)
    best_offer = _best_offer(offers)
    material_updates: dict[str, Any] = {}

    if offers:
        material_updates["lista_fornecedores"] = offers

    if update_payload.get("justificativa") not in (None, ""):
        material_updates["justificativa"] = update_payload["justificativa"]

    allow_direct_values = bool(offers) or bool(update_payload.get("fornecedor"))
    if allow_direct_values:
        for field in (
            "fornecedor",
            "valor_unitario",
            "valor_total",
            "preco_a_vista",
            "preco_a_prazo",
            "num_parcelas",
            "frete",
        ):
            value = update_payload.get(field)
            if value not in (None, ""):
                material_updates[field] = value

    if best_offer is not None:
        material_updates.setdefault("fornecedor", best_offer.fornecedor)
        material_updates.setdefault("valor_unitario", best_offer.valor_unitario)
        material_updates.setdefault("valor_total", best_offer.valor_total)
        material_updates.setdefault("preco_a_vista", best_offer.preco_a_vista)
        material_updates.setdefault("preco_a_prazo", best_offer.preco_a_prazo)
        material_updates.setdefault("num_parcelas", best_offer.num_parcelas)
        material_updates.setdefault("frete", best_offer.frete)

    return material.model_copy(update=material_updates)


def reason_about_material_suppliers(
    agent,
    area_name: str,
    material: MaterialObra,
    thread_id: str,
) -> dict:
    result = agent.invoke(
        {
            "messages": [
                SystemMessage(content=SUPPLIER_REASONING_SYSTEM_PROMPT),
                HumanMessage(content=_material_prompt(area_name, material)),
            ]
        },
        config={"configurable": {"thread_id": thread_id}},
    )
    return _extract_json_object(_message_text(result["messages"][-1]))


def preencher_fornecedores_com_reasoning_agent(
    lista_materiais: ListaMateriaisObra,
    reasoning_llm=None,
    supplier_search_tools: list | None = None,
    max_materials: int | None = None,
) -> ListaMateriaisObra:
    """Receive ListaMateriaisObra, run the ReAct reasoning agent, and fill supplier fields."""
    agent = build_supplier_reasoning_agent(
        reasoning_llm=reasoning_llm or llm,
        supplier_search_tools=supplier_search_tools,
        checkpointer=MemorySaver(),
    )

    processed = 0
    updated_areas = []
    for area_index, area in enumerate(lista_materiais.areas):
        updated_materials = []
        for material_index, material in enumerate(area.materiais):
            if max_materials is not None and processed >= max_materials:
                updated_materials.append(material)
                continue

            thread_id = f"supplier-reasoning-{area_index}-{material_index}-{hashlib.md5(material.nome.encode()).hexdigest()}"
            update_payload = reason_about_material_suppliers(
                agent=agent,
                area_name=area.area,
                material=material,
                thread_id=thread_id,
            )
            updated_materials.append(_apply_supplier_update(material, update_payload))
            processed += 1

        updated_areas.append(area.model_copy(update={"materiais": updated_materials}))

    return lista_materiais.model_copy(update={"areas": updated_areas})

In [ ]:
# Example, after you have a ListaMateriaisObra instance:
lista_com_fornecedores = preencher_fornecedores_com_reasoning_agent(
    lista_materiais=lista_materiais_quantificada,
    supplier_search_tools=[search_supplier_serp_tool, calculator_tool, percentage_tool, percent_change_tool],
    max_materials=3,
)
